In [1]:
import numpy as np
from cobra.util.array import create_stoichiometric_matrix
from docplex.mp.model import Model
from cobra.io import read_sbml_model
from operator import itemgetter


In [2]:
"""
Simple Test Implementation: MILP for Shortest Elementary Conversion Modes
Based on von Kamp & Klamt (2014) - Equations 1, 2, 7-11

This is conversions minimal working example to test the core MILP formulation.
"""

def is_combination(candidate_fluxes, previous_fluxes):
    if not previous_fluxes:
        return False

    compatible = [
        f for f in previous_fluxes
        if all(
            f[dim] == 0 if candidate_fluxes[dim] == 0
            else (f[dim] >= 0 if candidate_fluxes[dim] > 0 else f[dim] <= 0)
            for dim in range(len(candidate_fluxes))
        )
    ]

    if not compatible:
        return False

    m = len(compatible)
    check_mdl = Model()
    lam = check_mdl.continuous_var_list(m, lb=0, name="lam")

    for dim in range(len(candidate_fluxes)):
        target = candidate_fluxes[dim]

        # Skip trivially-zero dimensions — no information, no constraint needed
        if target == 0:
            continue

        lhs_coeffs = [compatible[j][dim] for j in range(m)]

        # If all compatible modes have zero flux here, combination is impossible
        if all(c == 0 for c in lhs_coeffs):
            check_mdl.end()
            return False

        check_mdl.add_constraint(
            check_mdl.sum(lam[j] * lhs_coeffs[j] for j in range(m)) == target
        )

    check_mdl.minimize(0)
    sol = check_mdl.solve(log_output=False)
    check_mdl.end()

    return sol is not None

def over_one(candidate_ecm):
    array = np.array(candidate_ecm)
    mask = (array != 0)

    return sum(mask)>1

def elementaries(modes):
    conversions= [ecm['conversions'] for ecm in modes]
    conv_copy = conversions.copy()
    true_ecms = []
    for i in range(len(conv_copy)):
        test_min = conversions.pop(i)
        if is_combination(test_min, conversions):
            conversions = conv_copy.copy()
        else:
            conversions = conv_copy.copy()
            true_ecms.append(modes[i])
    return true_ecms

def filter_message(ecms):
    current_ecms = len(ecms)
    ecms = elementaries(ecms)
    new_ecms = len(ecms)
    print(f"Filtering finished: Kept {new_ecms} ECMs out of {current_ecms} candidates")
    return ecms

def enumerate_k_shortest_ecms(N, exchanges, metabolites, k=5, reversible_pairs=None, beta=1000):
    m, n = N.shape
    n_ext = len(exchanges)
    ecms = []
    
    mdl = Model(name="k_Shortest_ECMs")
    
    # Reaction variables
    r = mdl.continuous_var_list(n, lb=0, name='r')
    z = mdl.binary_var_list(n, name='z')
    
    # External metabolite activity indicators
    y = mdl.binary_var_list(n_ext, name='y')
    
    # Steady state
    for i in range(m):
        mdl.add_constraint(
            mdl.sum(N[i, j] * r[j] for j in range(n)) == 0,
            ctname=f'steady_state_{i}'
        )
    
    # Reaction indicator constraints
    for i in range(n):
        mdl.add_indicator(z[i], r[i] == 0, active_value=0, name=f'ind_zero_{i}')
        mdl.add_indicator(z[i], r[i] >= 1, active_value=1, name=f'ind_active_{i}')
    
    # External metabolite indicator constraints
    for i, rxn_idx in enumerate(exchanges):
        mdl.add_indicator(y[i], r[rxn_idx] == 0, active_value=0, name=f'ext_zero_{i}')
        mdl.add_indicator(y[i], r[rxn_idx] >= 1, active_value=1, name=f'ext_active_{i}')
    
    # Reversible pairs
    if reversible_pairs:
        for fwd, bwd in reversible_pairs:
            mdl.add_constraint(z[fwd] + z[bwd] <= 1, ctname=f'rev_pair_{fwd}_{bwd}')
    
    # At least one conversion
    mdl.add_constraint(mdl.sum(y) >= 2, ctname='at_least_one')
    
    # Objective: minimize active reactions AND active external metabolites
    mdl.minimize(mdl.sum(z) + beta * mdl.sum(y))
    
    found = 0
    iteration = 0
    current_size = 2
    
    while found < k:
        solution = mdl.solve(log_output=False)
        
        if solution is None:
            print(f"No more ECMs found after {found} ECMs\n"
                 "Performing final cleaning of candidate modes")
            ecms = filter_message(ecms)
            break
        
        r_values = [r[i].solution_value for i in range(n)]
        z_values = [int(z[i].solution_value) for i in range(n)]
        y_values = [int(y[i].solution_value) for i in range(n_ext)]
        
        ex_values = itemgetter(*exchanges)(r_values)
        
        active_ex = set()
        conversions = np.zeros(m)
        for idx, flux in enumerate(ex_values):
            if flux != 0:
                rxn_col = exchanges[idx]
                ext_met = np.flatnonzero(N[:, rxn_col])
                active_ex.update(ext_met)
                for met in ext_met:
                    conversions[met] = N[met, rxn_col] * flux
        active_ex = sorted(list(active_ex))
        active_ex_ids = [metabolites[idx] for idx in active_ex]
        size = len(active_ex)

        #When conversions bigger ecm is found, clean previously found ones
        if size > current_size:
            print(
                f"All candidate modes of size {current_size} found \n"
                "Filtering candidate modes to keep elementarity"
            )
            current_size = size
            ecms = filter_message(ecms)           

        constraint_expr = mdl.sum(z_values[i] * z[i] for i in range(n)) 
        # Add small epsilon to avoid numerical issues
        mdl.add_constraint(
            constraint_expr <= sum(z_values) - 1,
            ctname=f'exclusion_{iteration}'
        )
        iteration += 1
        
        existing_ecms = [ecm['conversions'] for ecm in ecms]
        if is_combination(conversions, existing_ecms) or not over_one(conversions):
            continue
        
        ecm = {
            'conversion_size': size,
            'conversions': conversions,
            'converted_mets': active_ex_ids,
            'fluxes': r_values,
            'exchange_fluxes': ex_values,
            'active_reactions': [i for i, val in enumerate(z_values) if val == 1],
            'active_exchanges': [exchanges[i] for i, val in enumerate(y_values) if val == 1],
            'size': sum(z_values),
            'exchange_size': sum(y_values)
        }
        ecms.append(ecm)
        found += 1
        print(f"Potential ECM {found} found: reactions={ecm['size']}, exchanges={ecm['exchange_size']}, "
              f"converted_mets={ecm['converted_mets']}")
    
    return ecms

In [3]:
model_name = 'M_model'
model = read_sbml_model('../../models/' + model_name + ".xml")
S = create_stoichiometric_matrix(model)

n_original = len(model.reactions)
fwd = [index for index, reaction in enumerate(model.reactions) if reaction.reversibility]

rev_pairs = []
for rev_off, i in enumerate(fwd):
    bwd_idx = n_original + rev_off
    S = np.append(S, np.transpose([-S[:, i]]), axis=1)
    rev_pairs.append((i, bwd_idx))

exchanges = [0,1,4,5]

mets = [met.id for met in model.metabolites]


ecms_M = enumerate_k_shortest_ecms(S,  exchanges, mets, k=5020, reversible_pairs=rev_pairs)

Potential ECM 1 found: reactions=3, exchanges=2, converted_mets=['A_c', 'B_c']
No more ECMs found after 1 ECMs
Performing final cleaning of candidate modes
Filtering finished: Kept 1 ECMs out of 1 candidates


In [4]:
model_name = 'PQS_model'
model = read_sbml_model('../../models/' + model_name + ".xml")
S = create_stoichiometric_matrix(model)

n_original = len(model.reactions)
fwd = [index for index, reaction in enumerate(model.reactions) if reaction.reversibility]

rev_pairs = []
for rev_off, i in enumerate(fwd):
    bwd_idx = n_original + rev_off
    S = np.append(S, np.transpose([-S[:, i]]), axis=1)
    rev_pairs.append((i, bwd_idx))

exchanges = [0, 1, 2, 3]

mets = [met.id for met in model.metabolites]

ecms_PQS = enumerate_k_shortest_ecms(S,  exchanges, mets, k=5020, reversible_pairs=rev_pairs)
ecms_PQS

Potential ECM 1 found: reactions=3, exchanges=2, converted_mets=['Q_c', 'S_c']
Potential ECM 2 found: reactions=3, exchanges=2, converted_mets=['P_c', 'S_c']
Potential ECM 3 found: reactions=4, exchanges=2, converted_mets=['B_c', 'S_c']
All candidate modes of size 2 found 
Filtering candidate modes to keep elementarity
Filtering finished: Kept 3 ECMs out of 3 candidates
No more ECMs found after 3 ECMs
Performing final cleaning of candidate modes
Filtering finished: Kept 3 ECMs out of 3 candidates


[{'conversion_size': 2,
  'conversions': array([ 0.,  0.,  0.,  0., -1.,  1.]),
  'converted_mets': ['Q_c', 'S_c'],
  'fluxes': [1.0, 1.0, 0, 0, 0, 0, 0, 1.0, 0, 0, 0, 0],
  'exchange_fluxes': (1.0, 1.0, 0, 0),
  'active_reactions': [0, 1, 7],
  'active_exchanges': [0, 1],
  'size': 3,
  'exchange_size': 2},
 {'conversion_size': 2,
  'conversions': array([ 0.,  0.,  0., -1.,  0.,  1.]),
  'converted_mets': ['P_c', 'S_c'],
  'fluxes': [1.0, 0, 1.0, 0, 0, 0, 0, 0, 1.0, 0, 0, 0],
  'exchange_fluxes': (1.0, 0, 1.0, 0),
  'active_reactions': [0, 2, 8],
  'active_exchanges': [0, 2],
  'size': 3,
  'exchange_size': 2},
 {'conversion_size': 2,
  'conversions': array([ 0., -1.,  0.,  0.,  0.,  1.]),
  'converted_mets': ['B_c', 'S_c'],
  'fluxes': [1.0, 0, 0, 1.0, 1.0, 0, 0, 0, 0, 0, 1.0, 0],
  'exchange_fluxes': (1.0, 0, 0, 1.0),
  'active_reactions': [0, 3, 4, 10],
  'active_exchanges': [0, 3],
  'size': 4,
  'exchange_size': 2}]

In [6]:
model_name = 'ecoli5010_no_b'
model = read_sbml_model('../../models/' + model_name + ".xml")
S = create_stoichiometric_matrix(model)

n_original = len(model.reactions)
fwd = [index for index, reaction in enumerate(model.reactions) if reaction.reversibility]

rev_pairs = []
for rev_off, i in enumerate(fwd):
    bwd_idx = n_original + rev_off
    S = np.append(S, np.transpose([-S[:, i]]), axis=1)
    rev_pairs.append((i, bwd_idx))

exchanges = [rxn for rxn in model.reactions if "EX" in rxn.id]
exchanges_idx = [model.reactions.index(rxn) for rxn in exchanges]

mets = [met.id for met in model.metabolites]

ecms = enumerate_k_shortest_ecms(S, exchanges_idx, mets, k=200, reversible_pairs=rev_pairs)

Potential ECM 1 found: reactions=15, exchanges=2, converted_mets=['glc_D_e', 'lac_D_e']
All candidate modes of size 2 found 
Filtering candidate modes to keep elementarity
Filtering finished: Kept 1 ECMs out of 1 candidates
Potential ECM 2 found: reactions=18, exchanges=3, converted_mets=['co2_e', 'etoh_e', 'glc_D_e']
Potential ECM 3 found: reactions=28, exchanges=3, converted_mets=['etoh_e', 'glc_D_e', 'succ_e']
All candidate modes of size 3 found 
Filtering candidate modes to keep elementarity
Filtering finished: Kept 3 ECMs out of 3 candidates
Potential ECM 4 found: reactions=23, exchanges=4, converted_mets=['ac_e', 'etoh_e', 'for_e', 'glc_D_e']
Potential ECM 5 found: reactions=25, exchanges=4, converted_mets=['ac_e', 'glc_D_e', 'h2_e', 'succ_e']
Potential ECM 6 found: reactions=25, exchanges=4, converted_mets=['ac_e', 'etoh_e', 'glc_D_e', 'succ_e']
Potential ECM 7 found: reactions=27, exchanges=4, converted_mets=['etoh_e', 'for_e', 'glc_D_e', 'succ_e']
Potential ECM 8 found: reacti

In [7]:
len(ecms)

60

In [8]:
#Now the enumeration is compared to results from ecmtool, requires running ecmtool.sh with the ecoli5010_no_b.xml model

conversions = [[ecms[i]['converted_mets'],ecms[i]['conversions'][ecms[i]['conversions'].nonzero()]] for i in range (len(ecms))]
conversions

[[['glc_D_e', 'lac_D_e'], array([ 1., -2.])],
 [['co2_e', 'etoh_e', 'glc_D_e'], array([-2., -2.,  1.])],
 [['etoh_e', 'glc_D_e', 'succ_e'], array([-3.,  5., -6.])],
 [['ac_e', 'etoh_e', 'for_e', 'glc_D_e'], array([-1., -1., -2.,  1.])],
 [['ac_e', 'glc_D_e', 'h2_e', 'succ_e'], array([-1.,  1., -1., -1.])],
 [['ac_e', 'etoh_e', 'glc_D_e', 'succ_e'], array([-1., -1.,  2., -2.])],
 [['etoh_e', 'for_e', 'glc_D_e', 'succ_e'], array([-3. , -5. ,  2.5, -1. ])],
 [['etoh_e', 'glc_D_e', 'h2_e', 'succ_e'], array([-1.,  5., -5., -7.])],
 [['co2_e', 'etoh_e', 'glc_D_e', 'h2_e'], array([-7. , -4. ,  2.5, -6. ])],
 [['co2_e', 'etoh_e', 'for_e', 'glc_D_e'], array([-1. , -4. , -6. ,  2.5])],
 [['etoh_e', 'glc_D_e', 'nh4_e', 'succ_e'],
  array([-198.82844828,  383.93103448,  107.77586207, -378.70862069])],
 [['ac_e', 'co2_e', 'etoh_e', 'glc_D_e', 'h2_e'],
  array([-1., -2., -1.,  1., -2.])],
 [['co2_e', 'etoh_e', 'glc_D_e', 'h2_e', 'nh4_e'],
  array([-309.25862069, -306.86206897,  219.01293103,  -23.74

In [9]:
len(conversions)

60

In [10]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional


# ── Data structures ────────────────────────────────────────────────────────────

@dataclass
class ECM:
    """Sparse ECM: only stores non-zero metabolite→value pairs."""
    rates: dict  # {metabolite_name: float}

    @classmethod
    def from_list(cls, mets: list[str], vals: np.ndarray) -> "ECM":
        return cls(rates=dict(zip(mets, vals)))

    def to_dense(self, metabolites: list[str], tol: float = 1e-10) -> np.ndarray:
        vec = np.array([self.rates.get(m, 0.0) for m in metabolites])
        return vec

    def __repr__(self):
        items = ", ".join(f"{m}: {v:.4g}" for m, v in self.rates.items())
        return f"ECM({{{items}}})"


# ── Normalisation ──────────────────────────────────────────────────────────────

def normalize(vec: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    """
    Canonical form for comparison:
      1. Divide by L2 norm  →  scale-invariant
      2. Flip sign so the first significant element is positive  →  sign-invariant
    """
    norm = np.linalg.norm(vec)
    if norm < tol:
        return vec
    v = vec / norm
    for x in v:
        if abs(x) > tol:
            if x < 0:
                v = -v
            break
    return v


# ── Loaders ────────────────────────────────────────────────────────────────────

def load_ecms_from_list(raw: list) -> list[ECM]:
    """Convert your list-of-lists into ECM objects."""
    return [ECM.from_list(mets, vals) for mets, vals in raw]


def load_ecms_from_csv(
    path: str,
    sep: str = ",",
    metabolite_mapping: Optional[dict[str, str]] = None,
) -> tuple[list[ECM], list[str]]:
    df = pd.read_csv(path, sep=sep)
    
    # Strip "M_" prefix: keep everything after index 2
    df.columns = [col[2:] if col.startswith("M_") else col for col in df.columns]
    
    if metabolite_mapping:
        df.rename(columns=metabolite_mapping, inplace=True)
    metabolites = list(df.columns)
    ecms = [
        ECM(rates={m: v for m, v in zip(metabolites, row) if abs(v) > 1e-12})
        for row in df.values.astype(float)
    ]
    return ecms, metabolites


# ── Comparison ─────────────────────────────────────────────────────────────────

@dataclass
class ComparisonResult:
    matched_pairs: list[tuple[int, int]] = field(default_factory=list)   # (list_idx, csv_idx)
    unmatched_list: list[int]            = field(default_factory=list)
    unmatched_csv:  list[int]            = field(default_factory=list)

    def summary(self, list_ecms, csv_ecms):
        print(f"✓  Matched   : {len(self.matched_pairs)}")
        print(f"✗  Only in list-of-lists : {len(self.unmatched_list)}")
        for i in self.unmatched_list:
            print(f"     [{i}] {list_ecms[i]}")
        print(f"✗  Only in CSV           : {len(self.unmatched_csv)}")
        for j in self.unmatched_csv:
            print(f"     [{j}] {csv_ecms[j]}")


def compare_ecm_sets(
    list_ecms:   list[ECM],
    csv_ecms:    list[ECM],
    metabolites: list[str],
    atol:        float = 1e-6,
) -> ComparisonResult:
    """
    Compare two sets of ECMs up to scaling and sign.
    Each ECM is normalised before comparison; atol controls the match tolerance.
    """
    list_norm = [normalize(e.to_dense(metabolites)) for e in list_ecms]
    csv_norm  = [normalize(e.to_dense(metabolites)) for e in csv_ecms]

    matched_pairs  = []
    used_csv       = set()

    for i, lv in enumerate(list_norm):
        for j, cv in enumerate(csv_norm):
            if j in used_csv:
                continue
            if np.allclose(lv, cv, atol=atol):
                matched_pairs.append((i, j))
                used_csv.add(j)
                break

    matched_list = {i for i, _ in matched_pairs}
    return ComparisonResult(
        matched_pairs  = matched_pairs,
        unmatched_list = [i for i in range(len(list_ecms)) if i not in matched_list],
        unmatched_csv  = [j for j in range(len(csv_ecms))  if j not in used_csv],
    )


# ── Entry point ────────────────────────────────────────────────────────────────

def compare(
    raw_list:            list,
    csv_path:            str,
    metabolite_mapping:  Optional[dict[str, str]] = None,
    csv_sep:             str   = ",",   # ← was "\t"
    atol:                float = 1e-6,
) -> ComparisonResult:
    """
    Full pipeline:
      raw_list           – your list-of-lists
      csv_path           – path to the CSV/TSV file
      metabolite_mapping – {csv_col: canonical_name}, e.g. {"M_M1": "glc_D_e"}
      csv_sep            – column separator in the CSV file
      atol               – absolute tolerance for floating-point comparison
    """
    list_ecms            = load_ecms_from_list(raw_list)
    csv_ecms, metabolites = load_ecms_from_csv(csv_path, sep=csv_sep,
                                               metabolite_mapping=metabolite_mapping)

    # All metabolites present in either source
    all_mets_in_list = {m for e in list_ecms for m in e.rates}
    metabolites      = list(dict.fromkeys(metabolites + sorted(all_mets_in_list - set(metabolites))))

    result = compare_ecm_sets(list_ecms, csv_ecms, metabolites, atol=atol)
    result.summary(list_ecms, csv_ecms)
    return result

In [11]:
result = compare(conversions, "../../results/ecoli5010.csv")

✓  Matched   : 60
✗  Only in list-of-lists : 0
✗  Only in CSV           : 1
     [60] ECM({})
